## **Подготовка инструментов для нечеткой свертки с лингвистическими переменными**

In [ ]:
class LinguisticVariable:

  """Описание лингвистических переменных"""
  def __init__(self, name, universe_min, universe_max):
      self.name = name
      self.universe_min = universe_min
      self.universe_max = universe_max
      self.terms = {}

  def add_term(self, term_name, mf_type, params):
      self.terms[term_name] = {'type': mf_type, 'params': params}

  def fuzzify(self, x, term_name):
      if term_name not in self.terms:
          raise ValueError(f"Терм '{term_name}' не найден")
      term = self.terms[term_name]
      if term['type'] == 'trimf':
          return trimf(x, *term['params'])
      elif term['type'] == 'trapmf':
          return trapmf(x, *term['params'])
      return 0.0

  def get_all_memberships(self, x):
      return {term: self.fuzzify(x, term) for term in self.terms}

  def plot(self, ax=None, points=1000):
      if ax is None:
          ax = plt.gca()
      x_vals = np.linspace(self.universe_min, self.universe_max, points)
      for term_name in self.terms:
          y_vals = [self.fuzzify(x, term_name) for x in x_vals]
          ax.plot(x_vals, y_vals, label=term_name, linewidth=2)
      ax.set_xlabel(self.name)
      ax.set_ylabel('Степень принадлежности')
      ax.set_title(self.name)
      ax.legend()
      ax.grid(True, alpha=0.3)
      return ax

In [ ]:
def trimf(x, a, b, c):
  """Треугольная функция"""
  if x <= a or x >= c:
    return 0.0
  elif a < x <= b:
    return (x - a) / (b - a) if b != a else 1.0
  else:
    return (c - x) / (c - b) if c != b else 1.0


def trapmf(x, a, b, c, d):
  """Трапецевидная функция"""
  if x <= a or x >= d:
      return 0.0
  elif a < x <= b:
      return (x - a) / (b - a) if b != a else 1.0
  elif b < x <= c:
      return 1.0
  else:
      return (d - x) / (d - c) if d != c else 1.0

In [ ]:
class FuzzySystem:

  """Описание нечеткой свертки с лингвистическими переменными"""
  def __init__(self):
      self.inputs = {}
      self.output = None
      self.rules = []

  def add_input(self, name, linguistic_var):
      self.inputs[name] = linguistic_var

  def set_output(self, name, linguistic_var):
      self.output = (name, linguistic_var)

  def add_rule(self, premise, consequent):
      self.rules.append({'premise': premise, 'consequent': consequent})

  def compute(self, inputs_values):
      if self.output is None:
          raise ValueError("Выходная переменная не установлена")

      # Фаззификация
      memberships = {}
      for inp_name, inp_var in self.inputs.items():
          x = inputs_values[inp_name]
          memberships[inp_name] = inp_var.get_all_memberships(x)

      # Активация правил
      rule_activations = []
      for rule in self.rules:
          activations = []
          for inp_name, term_name in rule['premise'].items():
              if inp_name in memberships:
                  mu = memberships[inp_name].get(term_name, 0)
                  activations.append(mu)
          if activations:
              rule_strength = min(activations)
              rule_activations.append({
                  'strength': rule_strength,
                  'consequent': rule['consequent']})

      # Агрегация
      output_name, output_var = self.output
      aggregated = {term: 0.0 for term in output_var.terms}
      for activation in rule_activations:
          strength = activation['strength']
          term_name = activation['consequent']
          if strength > aggregated.get(term_name, 0):
              aggregated[term_name] = strength

      # Дефаззификация
      return self._defuzzify(aggregated, output_var)

  def _defuzzify(self, aggregated, output_var):
      points = 1000
      x_vals = np.linspace(output_var.universe_min, output_var.universe_max, points)
      numerator = 0.0
      denominator = 0.0

      for x in x_vals:
          membership_values = []
          for term_name, agg_val in aggregated.items():
              if agg_val > 0:
                  mf_val = output_var.fuzzify(x, term_name)
                  membership_values.append(min(agg_val, mf_val))
          y = max(membership_values) if membership_values else 0.0
          numerator += x * y
          denominator += y

      return numerator / denominator if denominator > 0 else 0.0

In [ ]:
def apply_fuzzy_convolution(df):
  """Применение свертки к подготовленным данным"""
  return df.apply(
      lambda row: fuzzy_system.compute(row.to_dict()),
      axis=1
      ).values

## **Выгрузка исходных данных**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Загружаем данные
df = pd.read_csv('smolensk_dataset (1).csv', sep=';')

# Очищаем названия колонок
df.columns = df.columns.str.strip()
df

In [ ]:
df.info()

## **Современная городская среда**

In [ ]:
# Выделяем sub_df для современной городской среды
urban_city_area = df.iloc[:, 4:7]
urban_city_area.rename(columns={'Исполнение бюджета. %': 'budget',
    'Благоустроенные дворы. ед.': 'yards',
    'Удовлетворенность городской средой. %': 'satisfaction'}, inplace=True)
urban_city_area

In [ ]:
# Назначение лингвистических переменных
# Исполнение бюджета (5 термов)
budget_var = LinguisticVariable('Исполнение бюджета, %', 60, 100)
budget_var.add_term('очень_низкое', 'trimf', [60, 63, 68])      # 60-68% - критически низкое
budget_var.add_term('низкое', 'trimf', [65, 70, 75])            # 65-75% - низкое
budget_var.add_term('среднее', 'trimf', [72, 78, 84])           # 72-84% - среднее
budget_var.add_term('высокое', 'trimf', [80, 86, 92])           # 80-92% - высокое
budget_var.add_term('очень_высокое', 'trimf', [88, 94, 100])    # 88-100% - очень высокое

# Благоустроенные дворы (5 термов)
yards_var = LinguisticVariable('Благоустроенные дворы, ед.', 10, 70)
yards_var.add_term('очень_мало', 'trimf', [10, 15, 22])         # 10-22 ед. - критически мало
yards_var.add_term('мало', 'trimf', [18, 25, 32])               # 18-32 ед. - мало
yards_var.add_term('средне', 'trimf', [28, 35, 42])             # 28-42 ед. - средне
yards_var.add_term('много', 'trimf', [38, 45, 52])              # 38-52 ед. - много
yards_var.add_term('очень_много', 'trimf', [48, 55, 70])        # 48-70 ед. - очень много

# Удовлетворенность (5 термов)
satisfaction_var = LinguisticVariable('Удовлетворенность городской средой, %', 30, 60)
satisfaction_var.add_term('очень_низкая', 'trimf', [30, 33, 37])   # 30-37% - критически низкая
satisfaction_var.add_term('низкая', 'trimf', [34, 38, 42])         # 34-42% - низкая
satisfaction_var.add_term('средняя', 'trimf', [39, 43, 47])        # 39-47% - средняя
satisfaction_var.add_term('высокая', 'trimf', [44, 48, 52])        # 44-52% - высокая
satisfaction_var.add_term('очень_высокая', 'trimf', [49, 53, 60])  # 49-60% - очень высокая

# Индекс качества современной городской среды (6 термов)
quality_var = LinguisticVariable('Индекс качества СГС', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])            # 1-20 - полный провал
quality_var.add_term('плохое', 'trimf', [15, 28, 40])              # 15-40 - плохо
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])  # 35-65 - удовлетворительно
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])             # 60-84 - хорошо
quality_var.add_term('отличное', 'trimf', [78, 88, 95])            # 78-95 - отлично
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])       # 90-100 - превосходно

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('yards', yards_var)
fuzzy_system.add_input('satisfaction', satisfaction_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # бюджет: очень низкий
    ({'budget': 'очень_низкое', 'yards': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'очень_мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'очень_мало', 'satisfaction': 'средняя'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    ({'budget': 'очень_низкое', 'yards': 'мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    ({'budget': 'очень_низкое', 'yards': 'средне', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'yards': 'средне', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'средне', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'средне', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'средне', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'очень_низкое', 'yards': 'много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'много', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'yards': 'много', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'очень_низкое', 'yards': 'очень_много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'очень_много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'yards': 'очень_много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'yards': 'очень_много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'yards': 'очень_много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # бюджет: низкий
    ({'budget': 'низкое', 'yards': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'yards': 'очень_мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'yards': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    ({'budget': 'низкое', 'yards': 'мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'yards': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'низкое', 'yards': 'средне', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'средне', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'средне', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'средне', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'yards': 'средне', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'низкое', 'yards': 'много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'yards': 'много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'yards': 'много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'низкое', 'yards': 'очень_много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'yards': 'очень_много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'yards': 'очень_много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'yards': 'очень_много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'низкое', 'yards': 'очень_много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # бюджет: средний
    ({'budget': 'среднее', 'yards': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'среднее', 'yards': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'среднее', 'yards': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'среднее', 'yards': 'средне', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'среднее', 'yards': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'средне', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'средне', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'средне', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'среднее', 'yards': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'среднее', 'yards': 'много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'среднее', 'yards': 'очень_много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'очень_много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'yards': 'очень_много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'среднее', 'yards': 'очень_много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'среднее', 'yards': 'очень_много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # бюджет: высокий
    ({'budget': 'высокое', 'yards': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'высокое', 'yards': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'высокое', 'yards': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'высокое', 'yards': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'высокое', 'yards': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'высокое', 'yards': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'высокое', 'yards': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'высокое', 'yards': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'мало', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'высокое', 'yards': 'средне', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'средне', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'средне', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'высокое', 'yards': 'средне', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'высокое', 'yards': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'высокое', 'yards': 'много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'высокое', 'yards': 'много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    ({'budget': 'высокое', 'yards': 'очень_много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'yards': 'очень_много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'высокое', 'yards': 'очень_много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'высокое', 'yards': 'очень_много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'высокое', 'yards': 'очень_много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # бюджет: очень высокий
    ({'budget': 'очень_высокое', 'yards': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_высокое', 'yards': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'yards': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_высокое', 'yards': 'очень_мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    ({'budget': 'очень_высокое', 'yards': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'yards': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'yards': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'мало', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    ({'budget': 'очень_высокое', 'yards': 'средне', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'средне', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'средне', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'средне', 'satisfaction': 'очень_высокая'}, 'отличное'),

    ({'budget': 'очень_высокое', 'yards': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'yards': 'много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'очень_высокое', 'yards': 'много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    ({'budget': 'очень_высокое', 'yards': 'очень_много', 'satisfaction': 'очень_низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'очень_много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'yards': 'очень_много', 'satisfaction': 'средняя'}, 'отличное'),
    ({'budget': 'очень_высокое', 'yards': 'очень_много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'очень_высокое', 'yards': 'очень_много', 'satisfaction': 'очень_высокая'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
urban_city_area['Индекс качества современной городской среды'] = apply_fuzzy_convolution(urban_city_area)
urban_city_area

In [ ]:
urban_city_area.describe()

### Графики для "Современная городская среда"

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# budget_var.plot(ax=axes[0, 0])
# # Добавляем вертикальные линии для основных квантилей
# for q in [0.25, 0.5, 0.75]:
#     val = df['Исполнение бюджета. %'].quantile(q)
#     axes[0, 0].axvline(val, color='gray', linestyle='--', alpha=0.5,
#                        label=f'Q{q:.0%}: {val:.1f}')

# yards_var.plot(ax=axes[0, 1])
# for q in [0.25, 0.5, 0.75]:
#     val = df['Благоустроенные дворы. ед.'].quantile(q)
#     axes[0, 1].axvline(val, color='gray', linestyle='--', alpha=0.5,
#                        label=f'Q{q:.0%}: {val:.1f}')

# satisfaction_var.plot(ax=axes[1, 0])
# for q in [0.25, 0.5, 0.75]:
#     val = df['Удовлетворенность городской средой. %'].quantile(q)
#     axes[1, 0].axvline(val, color='gray', linestyle='--', alpha=0.5,
#                        label=f'Q{q:.0%}: {val:.1f}')

# quality_var.plot(ax=axes[1, 1])
# plt.tight_layout()

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# # Гистограммы
# for idx, col in enumerate(df.columns):
#     ax = axes[idx // 2, idx % 2]
#     ax.hist(df[col], bins=20, edgecolor='black', alpha=0.7)
#     ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Среднее: {df[col].mean():.2f}')
#     ax.axvline(df[col].median(), color='green', linestyle='--', label=f'Медиана: {df[col].median():.2f}')
#     ax.set_xlabel(col)
#     ax.set_ylabel('Частота')
#     ax.set_title(f'Распределение: {col}')
#     ax.legend()
#     ax.grid(True, alpha=0.3)

# plt.tight_layout()

In [ ]:
# fig = plt.figure(figsize=(20, 12))

# # Распределение индекса качества
# ax1 = plt.subplot(2, 2, 1)
# ax1.hist(df['Индекс_качества'], bins=25, edgecolor='black', alpha=0.7, color='skyblue')
# ax1.axvline(df['Индекс_качества'].mean(), color='red', linestyle='--', linewidth=2, label=f'Среднее: {df["Индекс_качества"].mean():.1f}')
# ax1.axvline(df['Индекс_качества'].median(), color='green', linestyle='--', linewidth=2, label=f'Медиана: {df["Индекс_качества"].median():.1f}')
# ax1.set_xlabel('Индекс качества')
# ax1.set_ylabel('Частота')
# ax1.set_title('Распределение индекса качества')
# ax1.legend()
# ax1.grid(True, alpha=0.3)

# # Тепловая карта корреляций
# ax5 = plt.subplot(2, 2, 2)
# corr = df[['Исполнение бюджета, %', 'Благоустроенные дворы, ед.',
#            'Удовлетворенность городской средой, %', 'Индекс_качества']].corr()
# im = ax5.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
# ax5.set_xticks(range(len(corr.columns)))
# ax5.set_yticks(range(len(corr.columns)))
# ax5.set_xticklabels(corr.columns, rotation=45, ha='right')
# ax5.set_yticklabels(corr.columns)
# for i in range(len(corr.columns)):
#     for j in range(len(corr.columns)):
#         ax5.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
#                 color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
# ax5.set_title('Корреляционная матрица')
# plt.colorbar(im, ax=ax5)

# # Вклад признаков
# ax8 = plt.subplot(2, 2, 3)
# feature_importance = pd.Series({
#     'Бюджет': df['Исполнение бюджета, %'].corr(df['Индекс_качества']),
#     'Дворы': df['Благоустроенные дворы, ед.'].corr(df['Индекс_качества']),
#     'Удовлетворенность': df['Удовлетворенность городской средой, %'].corr(df['Индекс_качества'])
# })
# feature_importance.sort_values().plot(kind='barh', color=['skyblue', 'lightgreen', 'salmon'], ax=ax8)
# ax8.set_xlabel('Корреляция с индексом качества')
# ax8.set_title('Вклад признаков в индекс качества')
# ax8.grid(True, alpha=0.3)

# # Плотность распределения
# ax9 = plt.subplot(2, 2, 4)
# for col in df.columns[:3]:
#     sns.kdeplot(df[col], ax=ax9, label=col, linewidth=2)
# ax9.set_xlabel('Значение')
# ax9.set_ylabel('Плотность')
# ax9.set_title('Плотность распределения признаков')
# ax9.legend()
# ax9.grid(True, alpha=0.3)

# plt.tight_layout()

## **Дорожно-транспортный комплекс**

In [ ]:
road_transport_complex = df.iloc[:, 7:16]
road_transport_complex

In [ ]:
road_transport_complex.describe()

In [ ]:
road_transport_complex_r1 = road_transport_complex.drop(
    ['Дороги в нормативном состоянии. %',
     'Исполнение бюджета. %',
     'Регулируемые переходы. ед.'], axis=1)
road_transport_complex_r1.rename(columns={
    'Отремонтированные дороги. км': 'roads',
    'Пассажиропоток. тыс. поездок': 'passenger',
    'Рейсы по расписанию. %': 'schedule',
    'Средняя скорость на магистралях. км/ч': 'speed',
    'ДТП на 10 тыс. жителей': 'accident',
    'Срок устранения дефектов. суток': 'defect',
}, inplace=True)
road_transport_complex_r1

In [ ]:
road_transport_complex_r1.describe()

In [ ]:
# 1.1. Отремонтированные дороги (3 состояния)
roads_var = LinguisticVariable('Отремонтированные дороги, км', 15, 75)
roads_var.add_term('мало', 'trimf', [15, 20, 35])           # 15-35 - мало
roads_var.add_term('средне', 'trimf', [30, 45, 55])         # 30-55 - средне
roads_var.add_term('много', 'trimf', [45, 60, 75])          # 45-75 - много

# 1.2. Пассажиропоток (3 состояния)
passenger_var = LinguisticVariable('Пассажиропоток, тыс. поездок', 80, 245)
passenger_var.add_term('низкий', 'trimf', [80, 95, 130])     # 80-130 - низкий
passenger_var.add_term('средний', 'trimf', [120, 155, 185])  # 120-185 - средний
passenger_var.add_term('высокий', 'trimf', [170, 200, 245])  # 170-245 - высокий

# 1.3. Рейсы по расписанию (3 состояния)
schedule_var = LinguisticVariable('Рейсы по расписанию, %', 48, 73)
schedule_var.add_term('низкое', 'trimf', [48, 51, 57])      # 48-57 - низкое
schedule_var.add_term('среднее', 'trimf', [55, 60, 65])     # 55-65 - среднее
schedule_var.add_term('высокое', 'trimf', [62, 67, 73])     # 62-73 - высокое

# 1.4. Средняя скорость (3 состояния)
speed_var = LinguisticVariable('Средняя скорость, км/ч', 16, 24)
speed_var.add_term('низкая', 'trimf', [16, 17, 19])         # 16-19 - низкая
speed_var.add_term('средняя', 'trimf', [18, 20, 22])        # 18-22 - средняя
speed_var.add_term('высокая', 'trimf', [21, 22.5, 24])      # 21-24 - высокая

# 1.5. ДТП на 10 тыс. жителей (3 состояния) - ИНВЕРТИРОВАННЫЙ
accident_var = LinguisticVariable('ДТП на 10 тыс. жителей', 22, 44)
accident_var.add_term('низкое', 'trimf', [22, 24, 28])      # 22-28 - мало ДТП (хорошо)
accident_var.add_term('среднее', 'trimf', [26, 31, 36])     # 26-36 - средне
accident_var.add_term('высокое', 'trimf', [33, 38, 44])     # 33-44 - много ДТП (плохо)

# 1.6. Срок устранения дефектов (3 состояния) - ИНВЕРТИРОВАННЫЙ
defect_var = LinguisticVariable('Срок устранения дефектов, сут', 20, 46)
defect_var.add_term('быстро', 'trimf', [20, 22, 28])        # 20-28 - быстро (хорошо)
defect_var.add_term('средне', 'trimf', [26, 31, 37])        # 26-37 - средне
defect_var.add_term('медленно', 'trimf', [34, 40, 46])      # 34-46 - медленно (плохо)

# 1.7. Выход: ИНДЕКС КАЧЕСТВА ДОРОГ (6 термов)
quality_var = LinguisticVariable('Индекс качества дорог', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('roads', roads_var)
fuzzy_system.add_input('passenger', passenger_var)
fuzzy_system.add_input('speed', speed_var)
fuzzy_system.add_input('schedule', schedule_var)
fuzzy_system.add_input('accident', accident_var)
fuzzy_system.add_input('defect', defect_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
    # ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

# ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
road_transport_complex_r1['Индекс качества дорог'] = apply_fuzzy_convolution(road_transport_complex_r1)
road_transport_complex_r1

In [ ]:
road_transport_complex_r1.describe()

In [ ]:
road_transport_complex_r2 = road_transport_complex[['Дороги в нормативном состоянии. %',
                                                    'Исполнение бюджета. %',
                                                    'Регулируемые переходы. ед.']]
road_transport_complex_r2.rename(columns={
    'Исполнение бюджета. %': 'budget',
    'Дороги в нормативном состоянии. %': 'norm',
    'Регулируемые переходы. ед.': 'crossing'}, inplace=True)
road_transport_complex_r2

In [ ]:
road_transport_complex_r2.describe()

In [ ]:
# 1. Дороги в нормативном состоянии (40.9-52.3%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
norm_var = LinguisticVariable('Дороги в нормативном состоянии, %', 40, 53)
norm_var.add_term('очень_низкое', 'trimf', [40, 41.5, 43.5])      # 40-43.5 - очень низкое
norm_var.add_term('низкое', 'trimf', [42, 44, 46])                # 42-46 - низкое
norm_var.add_term('среднее', 'trimf', [44.5, 46.5, 48.5])         # 44.5-48.5 - среднее
norm_var.add_term('высокое', 'trimf', [47, 49, 51])               # 47-51 - высокое
norm_var.add_term('очень_высокое', 'trimf', [49.5, 51.5, 53])     # 49.5-53 - очень высокое

# 2. Исполнение бюджета (69.3-94.5%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
budget_var = LinguisticVariable('Исполнение бюджета, %', 68, 96)
budget_var.add_term('очень_низкий', 'trimf', [68, 70.5, 75])       # 68-75 - очень низкий
budget_var.add_term('низкий', 'trimf', [73, 77, 81])               # 73-81 - низкий
budget_var.add_term('средний', 'trimf', [79, 83, 87])              # 79-87 - средний
budget_var.add_term('высокий', 'trimf', [85, 88.5, 92])            # 85-92 - высокий
budget_var.add_term('очень_высокий', 'trimf', [90, 93, 96])        # 90-96 - очень высокий

# 3. Регулируемые переходы (11-38 ед.) - ЧЕМ БОЛЬШЕ, ТЕМ ЛУЧШЕ
crossing_var = LinguisticVariable('Регулируемые переходы, ед.', 10, 40)
crossing_var.add_term('очень_мало', 'trimf', [10, 13, 18])         # 10-18 - очень мало
crossing_var.add_term('мало', 'trimf', [15, 20, 25])               # 15-25 - мало
crossing_var.add_term('средне', 'trimf', [22, 27, 32])             # 22-32 - средне
crossing_var.add_term('много', 'trimf', [28, 33, 36])              # 28-36 - много
crossing_var.add_term('очень_много', 'trimf', [34, 37, 40])        # 34-40 - очень много

# 4. ИНДЕКС БЛАГОПОЛУЧИЯ ДОРОГ
quality_var = LinguisticVariable('Индекс качества дорог', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('norm', norm_var)
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('crossing', crossing_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # ==========================================
    # 1. НОРМАТИВНОСТЬ: ОЧЕНЬ НИЗКАЯ (40-43.5%)
    # ==========================================
    # 1.1. Бюджет: ОЧЕНЬ НИЗКИЙ (68-75%)
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 1.2. Бюджет: НИЗКИЙ (73-81%)
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 1.3. Бюджет: СРЕДНИЙ (79-87%)
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 1.4. Бюджет: ВЫСОКИЙ (85-92%)
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 1.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ (90-96%)
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 2. НОРМАТИВНОСТЬ: НИЗКАЯ (42-46%)
    # ==========================================
    # 2.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 2.2. Бюджет: НИЗКИЙ
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 2.3. Бюджет: СРЕДНИЙ
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 2.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # 2.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 3. НОРМАТИВНОСТЬ: СРЕДНЯЯ (44.5-48.5%)
    # ==========================================
    # 3.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 3.2. Бюджет: НИЗКИЙ
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 3.3. Бюджет: СРЕДНИЙ
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'очень_много'}, 'хорошее'),

    # 3.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # 3.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'отличное'),

    # ==========================================
    # 4. НОРМАТИВНОСТЬ: ВЫСОКАЯ (47-51%)
    # ==========================================
    # 4.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 4.2. Бюджет: НИЗКИЙ
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'хорошее'),

    # 4.3. Бюджет: СРЕДНИЙ
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'очень_много'}, 'отличное'),

    # 4.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'отличное'),

    # 4.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'превосходное'),

    # ==========================================
    # 5. НОРМАТИВНОСТЬ: ОЧЕНЬ ВЫСОКАЯ (49.5-53%)
    # ==========================================
    # 5.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'хорошее'),

    # 5.2. Бюджет: НИЗКИЙ
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'отличное'),

    # 5.3. Бюджет: СРЕДНИЙ
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'очень_много'}, 'отличное'),

    # 5.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'превосходное'),

    # 5.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'превосходное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
road_transport_complex_r2['Индекс благополучия дорог'] = apply_fuzzy_convolution(road_transport_complex_r2)
road_transport_complex_r2

In [ ]:
road_transport_complex_r2.describe()

## **Доступная среда**

In [ ]:
available_area = df.iloc[:, 16:19]
available_area.rename(columns={
    'Исполнение бюджета. %.1': 'budget',
    'Завершенные мероприятия. ед.': 'events',
    'Получатели адресной поддержки. чел.': 'support'
}, inplace=True)
available_area

In [ ]:
available_area.describe()

In [ ]:
# 1. Исполнение бюджета (66.1-95.3%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
budget_var = LinguisticVariable('Исполнение бюджета, %', 64, 97)
budget_var.add_term('очень_низкое', 'trimf', [64, 67, 72])           # 64-72% - критически низкое
budget_var.add_term('низкое', 'trimf', [70, 74.5, 79])               # 70-79% - низкое
budget_var.add_term('среднее', 'trimf', [77, 81.5, 86])              # 77-86% - среднее
budget_var.add_term('высокое', 'trimf', [84, 88.5, 93])              # 84-93% - высокое
budget_var.add_term('очень_высокое', 'trimf', [90.5, 94, 97])        # 90.5-97% - очень высокое

# 2. Завершенные мероприятия (4-16 ед.) - ЧЕМ БОЛЬШЕ, ТЕМ ЛУЧШЕ
events_var = LinguisticVariable('Завершенные мероприятия, ед.', 3, 17)
events_var.add_term('очень_мало', 'trimf', [3, 4.5, 6.5])            # 3-6.5 - критически мало
events_var.add_term('мало', 'trimf', [5.5, 7.5, 9.5])                # 5.5-9.5 - мало
events_var.add_term('средне', 'trimf', [8, 10, 12])                  # 8-12 - средне
events_var.add_term('много', 'trimf', [10.5, 12.5, 14.5])            # 10.5-14.5 - много
events_var.add_term('очень_много', 'trimf', [13, 15, 17])            # 13-17 - очень много

# 3. Получатели адресной поддержки (3056-5232 чел.) - ЧЕМ БОЛЬШЕ, ТЕМ ЛУЧШЕ
support_var = LinguisticVariable('Получатели адресной поддержки, чел.', 3000, 5300)
support_var.add_term('очень_мало', 'trimf', [3000, 3150, 3450])      # 3000-3450 - критически мало
support_var.add_term('мало', 'trimf', [3350, 3650, 3950])            # 3350-3950 - мало
support_var.add_term('средне', 'trimf', [3850, 4150, 4450])          # 3850-4450 - средне
support_var.add_term('много', 'trimf', [4350, 4650, 4950])           # 4350-4950 - много
support_var.add_term('очень_много', 'trimf', [4850, 5050, 5300])     # 4850-5300 - очень много

# 4. Выход: ИНДЕКС СОЦИАЛЬНОЙ ЭФФЕКТИВНОСТИ (6 термов, строго > 0)
quality_var = LinguisticVariable('Индекс социальной эффективности', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('events', events_var)
fuzzy_system.add_input('support', support_var)
fuzzy_system.set_output('quality', quality_var)

rules = [
    # ==========================================
    # 1. БЮДЖЕТ: ОЧЕНЬ НИЗКИЙ (64-72%)
    # ==========================================
    # 1.1. Мероприятия: ОЧЕНЬ МАЛО (3-6.5) + Поддержка: ОЧЕНЬ МАЛО (3000-3450)
    ({'budget': 'очень_низкое', 'events': 'очень_мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'очень_мало', 'support': 'мало'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'очень_мало', 'support': 'средне'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'очень_мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'очень_мало', 'support': 'очень_много'}, 'плохое'),

    # 1.2. Мероприятия: МАЛО (5.5-9.5)
    ({'budget': 'очень_низкое', 'events': 'мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'мало', 'support': 'мало'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'мало', 'support': 'очень_много'}, 'плохое'),

    # 1.3. Мероприятия: СРЕДНЕ (8-12)
    ({'budget': 'очень_низкое', 'events': 'средне', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'events': 'средне', 'support': 'мало'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'средне', 'support': 'средне'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'средне', 'support': 'много'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'средне', 'support': 'очень_много'}, 'удовлетворительное'),

    # 1.4. Мероприятия: МНОГО (10.5-14.5)
    ({'budget': 'очень_низкое', 'events': 'много', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'много', 'support': 'мало'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'много', 'support': 'средне'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'много', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'events': 'много', 'support': 'очень_много'}, 'удовлетворительное'),

    # 1.5. Мероприятия: ОЧЕНЬ МНОГО (13-17)
    ({'budget': 'очень_низкое', 'events': 'очень_много', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'очень_много', 'support': 'мало'}, 'плохое'),
    ({'budget': 'очень_низкое', 'events': 'очень_много', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'events': 'очень_много', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'events': 'очень_много', 'support': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 2. БЮДЖЕТ: НИЗКИЙ (70-79%)
    # ==========================================
    # 2.1. Мероприятия: ОЧЕНЬ МАЛО
    ({'budget': 'низкое', 'events': 'очень_мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'низкое', 'events': 'очень_мало', 'support': 'мало'}, 'катастрофа'),
    ({'budget': 'низкое', 'events': 'очень_мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'очень_мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'очень_мало', 'support': 'очень_много'}, 'плохое'),

    # 2.2. Мероприятия: МАЛО
    ({'budget': 'низкое', 'events': 'мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'низкое', 'events': 'мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'мало', 'support': 'очень_много'}, 'удовлетворительное'),

    # 2.3. Мероприятия: СРЕДНЕ
    ({'budget': 'низкое', 'events': 'средне', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'средне', 'support': 'мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'средне', 'support': 'средне'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'средне', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'events': 'средне', 'support': 'очень_много'}, 'удовлетворительное'),

    # 2.4. Мероприятия: МНОГО
    ({'budget': 'низкое', 'events': 'много', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'много', 'support': 'мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'много', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'events': 'много', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'events': 'много', 'support': 'очень_много'}, 'хорошее'),

    # 2.5. Мероприятия: ОЧЕНЬ МНОГО
    ({'budget': 'низкое', 'events': 'очень_много', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'низкое', 'events': 'очень_много', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'events': 'очень_много', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'events': 'очень_много', 'support': 'много'}, 'хорошее'),
    ({'budget': 'низкое', 'events': 'очень_много', 'support': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 3. БЮДЖЕТ: СРЕДНИЙ (77-86%)
    # ==========================================
    # 3.1. Мероприятия: ОЧЕНЬ МАЛО
    ({'budget': 'среднее', 'events': 'очень_мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'среднее', 'events': 'очень_мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'очень_мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'очень_мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'очень_мало', 'support': 'очень_много'}, 'удовлетворительное'),

    # 3.2. Мероприятия: МАЛО
    ({'budget': 'среднее', 'events': 'мало', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'мало', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'мало', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'мало', 'support': 'очень_много'}, 'удовлетворительное'),

    # 3.3. Мероприятия: СРЕДНЕ
    ({'budget': 'среднее', 'events': 'средне', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'среднее', 'events': 'средне', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'средне', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'средне', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'средне', 'support': 'очень_много'}, 'хорошее'),

    # 3.4. Мероприятия: МНОГО
    ({'budget': 'среднее', 'events': 'много', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'много', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'много', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'много', 'support': 'много'}, 'хорошее'),
    ({'budget': 'среднее', 'events': 'много', 'support': 'очень_много'}, 'хорошее'),

    # 3.5. Мероприятия: ОЧЕНЬ МНОГО
    ({'budget': 'среднее', 'events': 'очень_много', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'очень_много', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'events': 'очень_много', 'support': 'средне'}, 'хорошее'),
    ({'budget': 'среднее', 'events': 'очень_много', 'support': 'много'}, 'хорошее'),
    ({'budget': 'среднее', 'events': 'очень_много', 'support': 'очень_много'}, 'отличное'),

    # ==========================================
    # 4. БЮДЖЕТ: ВЫСОКИЙ (84-93%)
    # ==========================================
    # 4.1. Мероприятия: ОЧЕНЬ МАЛО
    ({'budget': 'высокое', 'events': 'очень_мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'высокое', 'events': 'очень_мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'высокое', 'events': 'очень_мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'высокое', 'events': 'очень_мало', 'support': 'много'}, 'плохое'),
    ({'budget': 'высокое', 'events': 'очень_мало', 'support': 'очень_много'}, 'удовлетворительное'),

    # 4.2. Мероприятия: МАЛО
    ({'budget': 'высокое', 'events': 'мало', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'высокое', 'events': 'мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'высокое', 'events': 'мало', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'мало', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'мало', 'support': 'очень_много'}, 'хорошее'),

    # 4.3. Мероприятия: СРЕДНЕ
    ({'budget': 'высокое', 'events': 'средне', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'средне', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'средне', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'средне', 'support': 'много'}, 'хорошее'),
    ({'budget': 'высокое', 'events': 'средне', 'support': 'очень_много'}, 'хорошее'),

    # 4.4. Мероприятия: МНОГО
    ({'budget': 'высокое', 'events': 'много', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'много', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'много', 'support': 'средне'}, 'хорошее'),
    ({'budget': 'высокое', 'events': 'много', 'support': 'много'}, 'хорошее'),
    ({'budget': 'высокое', 'events': 'много', 'support': 'очень_много'}, 'отличное'),

    # 4.5. Мероприятия: ОЧЕНЬ МНОГО
    ({'budget': 'высокое', 'events': 'очень_много', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'events': 'очень_много', 'support': 'мало'}, 'хорошее'),
    ({'budget': 'высокое', 'events': 'очень_много', 'support': 'средне'}, 'хорошее'),
    ({'budget': 'высокое', 'events': 'очень_много', 'support': 'много'}, 'отличное'),
    ({'budget': 'высокое', 'events': 'очень_много', 'support': 'очень_много'}, 'отличное'),

    # ==========================================
    # 5. БЮДЖЕТ: ОЧЕНЬ ВЫСОКИЙ (90.5-97%)
    # ==========================================
    # 5.1. Мероприятия: ОЧЕНЬ МАЛО
    ({'budget': 'очень_высокое', 'events': 'очень_мало', 'support': 'очень_мало'}, 'катастрофа'),
    ({'budget': 'очень_высокое', 'events': 'очень_мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'очень_высокое', 'events': 'очень_мало', 'support': 'средне'}, 'плохое'),
    ({'budget': 'очень_высокое', 'events': 'очень_мало', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'очень_мало', 'support': 'очень_много'}, 'удовлетворительное'),

    # 5.2. Мероприятия: МАЛО
    ({'budget': 'очень_высокое', 'events': 'мало', 'support': 'очень_мало'}, 'плохое'),
    ({'budget': 'очень_высокое', 'events': 'мало', 'support': 'мало'}, 'плохое'),
    ({'budget': 'очень_высокое', 'events': 'мало', 'support': 'средне'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'мало', 'support': 'много'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'мало', 'support': 'очень_много'}, 'хорошее'),

    # 5.3. Мероприятия: СРЕДНЕ
    ({'budget': 'очень_высокое', 'events': 'средне', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'средне', 'support': 'мало'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'средне', 'support': 'средне'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'средне', 'support': 'много'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'средне', 'support': 'очень_много'}, 'отличное'),

    # 5.4. Мероприятия: МНОГО
    ({'budget': 'очень_высокое', 'events': 'много', 'support': 'очень_мало'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'events': 'много', 'support': 'мало'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'много', 'support': 'средне'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'много', 'support': 'много'}, 'отличное'),
    ({'budget': 'очень_высокое', 'events': 'много', 'support': 'очень_много'}, 'отличное'),

    # 5.5. Мероприятия: ОЧЕНЬ МНОГО
    ({'budget': 'очень_высокое', 'events': 'очень_много', 'support': 'очень_мало'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'очень_много', 'support': 'мало'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'events': 'очень_много', 'support': 'средне'}, 'отличное'),
    ({'budget': 'очень_высокое', 'events': 'очень_много', 'support': 'много'}, 'отличное'),
    ({'budget': 'очень_высокое', 'events': 'очень_много', 'support': 'очень_много'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
available_area['Индекс удовлетворенности доступной среды'] = apply_fuzzy_convolution(available_area)
available_area

In [ ]:
available_area.describe()

## **Общественные пространства**

In [ ]:
public_spaces = df.iloc[:, 19:22]
public_spaces.rename(columns={
    'Исполнение бюджета. %.2': 'budget',
    'Благоустроенные общественные территории. ед.': 'territory',
    'Удовлетворенность городской средой. %.1': 'satisfaction'
}, inplace=True)
public_spaces

In [ ]:
public_spaces.describe()

In [ ]:
# 1. Исполнение бюджета (67.2-96.8%)
budget_var = LinguisticVariable('Исполнение бюджета, %', 65, 98)
budget_var.add_term('очень_низкое', 'trimf', [65, 68, 73])           # 65-73% - критически низкое
budget_var.add_term('низкое', 'trimf', [71, 75.5, 80])               # 71-80% - низкое
budget_var.add_term('среднее', 'trimf', [78, 82.5, 87])              # 78-87% - среднее
budget_var.add_term('высокое', 'trimf', [85, 89.5, 94])              # 85-94% - высокое
budget_var.add_term('очень_высокое', 'trimf', [91.5, 95, 98])        # 91.5-98% - очень высокое

# 2. Благоустроенные территории (8-23 ед.)
territory_var = LinguisticVariable('Благоустроенные территории, ед.', 7, 24)
territory_var.add_term('очень_мало', 'trimf', [7, 8.5, 11])          # 7-11 - критически мало
territory_var.add_term('мало', 'trimf', [10, 12.5, 15])              # 10-15 - мало
territory_var.add_term('средне', 'trimf', [13.5, 16, 18.5])          # 13.5-18.5 - средне
territory_var.add_term('много', 'trimf', [17, 19.5, 22])             # 17-22 - много
territory_var.add_term('очень_много', 'trimf', [20.5, 22.5, 24])     # 20.5-24 - очень много

# 3. Удовлетворенность городской средой (51.9-64.0%)
satisfaction_var = LinguisticVariable('Удовлетворенность, %', 51, 65)
satisfaction_var.add_term('очень_низкая', 'trimf', [51, 52.5, 54.5]) # 51-54.5 - очень низкая
satisfaction_var.add_term('низкая', 'trimf', [53.5, 55.5, 57.5])     # 53.5-57.5 - низкая
satisfaction_var.add_term('средняя', 'trimf', [56.5, 58.5, 60.5])    # 56.5-60.5 - средняя
satisfaction_var.add_term('высокая', 'trimf', [59.5, 61.5, 63.5])    # 59.5-63.5 - высокая
satisfaction_var.add_term('очень_высокая', 'trimf', [62.5, 64.5, 65]) # 62.5-65 - очень высокая

# 4. Выход: ИНДЕКС БЛАГОУСТРОЙСТВА (6 термов)
quality_var = LinguisticVariable('Индекс благоустройства', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('territory', territory_var)
fuzzy_system.add_input('satisfaction', satisfaction_var)
fuzzy_system.set_output('quality', quality_var)

rules = [
    # 1. БЮДЖЕТ: ОЧЕНЬ НИЗКИЙ (65-73%)
    # 1.1. Территории: ОЧЕНЬ МАЛО (7-11) + Удовлетворенность: ОЧЕНЬ НИЗКАЯ (51-54.5)
    ({'budget': 'очень_низкое', 'territory': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'очень_мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'очень_мало', 'satisfaction': 'средняя'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    # 1.2. Территории: МАЛО (10-15)
    ({'budget': 'очень_низкое', 'territory': 'мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    # 1.3. Территории: СРЕДНЕ (13.5-18.5)
    ({'budget': 'очень_низкое', 'territory': 'средне', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_низкое', 'territory': 'средне', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'средне', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'средне', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'средне', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 1.4. Территории: МНОГО (17-22)
    ({'budget': 'очень_низкое', 'territory': 'много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'много', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'territory': 'много', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 1.5. Территории: ОЧЕНЬ МНОГО (20.5-24)
    ({'budget': 'очень_низкое', 'territory': 'очень_много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'очень_много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_низкое', 'territory': 'очень_много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'territory': 'очень_много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_низкое', 'territory': 'очень_много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 2. БЮДЖЕТ: НИЗКИЙ (71-80%)
    # 2.1. Территории: ОЧЕНЬ МАЛО
    ({'budget': 'низкое', 'territory': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'territory': 'очень_мало', 'satisfaction': 'низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'territory': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'плохое'),

    # 2.2. Территории: МАЛО
    ({'budget': 'низкое', 'territory': 'мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'низкое', 'territory': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 2.3. Территории: СРЕДНЕ
    ({'budget': 'низкое', 'territory': 'средне', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'средне', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'средне', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'средне', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'territory': 'средне', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 2.4. Территории: МНОГО
    ({'budget': 'низкое', 'territory': 'много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'много', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'territory': 'много', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'territory': 'много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 2.5. Территории: ОЧЕНЬ МНОГО
    ({'budget': 'низкое', 'territory': 'очень_много', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'низкое', 'territory': 'очень_много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'territory': 'очень_много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'низкое', 'territory': 'очень_много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'низкое', 'territory': 'очень_много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 3. БЮДЖЕТ: СРЕДНИЙ (78-87%)
    # 3.1. Территории: ОЧЕНЬ МАЛО
    ({'budget': 'среднее', 'territory': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'среднее', 'territory': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 3.2. Территории: МАЛО
    ({'budget': 'среднее', 'territory': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 3.3. Территории: СРЕДНЕ
    ({'budget': 'среднее', 'territory': 'средне', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'среднее', 'territory': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'средне', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'средне', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'средне', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 3.4. Территории: МНОГО
    ({'budget': 'среднее', 'territory': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'много', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'среднее', 'territory': 'много', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 3.5. Территории: ОЧЕНЬ МНОГО
    ({'budget': 'среднее', 'territory': 'очень_много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'очень_много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'среднее', 'territory': 'очень_много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'среднее', 'territory': 'очень_много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'среднее', 'territory': 'очень_много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # 4. БЮДЖЕТ: ВЫСОКИЙ (85-94%)
    # 4.1. Территории: ОЧЕНЬ МАЛО
    ({'budget': 'высокое', 'territory': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'высокое', 'territory': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'высокое', 'territory': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'высокое', 'territory': 'очень_мало', 'satisfaction': 'высокая'}, 'плохое'),
    ({'budget': 'высокое', 'territory': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 4.2. Территории: МАЛО
    ({'budget': 'высокое', 'territory': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'высокое', 'territory': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'высокое', 'territory': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'мало', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 4.3. Территории: СРЕДНЕ
    ({'budget': 'высокое', 'territory': 'средне', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'средне', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'средне', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'высокое', 'territory': 'средне', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 4.4. Территории: МНОГО
    ({'budget': 'высокое', 'territory': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'много', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'высокое', 'territory': 'много', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'высокое', 'territory': 'много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # 4.5. Территории: ОЧЕНЬ МНОГО
    ({'budget': 'высокое', 'territory': 'очень_много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'высокое', 'territory': 'очень_много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'высокое', 'territory': 'очень_много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'высокое', 'territory': 'очень_много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'высокое', 'territory': 'очень_много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # 5. БЮДЖЕТ: ОЧЕНЬ ВЫСОКИЙ (91.5-98%)
    # 5.1. Территории: ОЧЕНЬ МАЛО
    ({'budget': 'очень_высокое', 'territory': 'очень_мало', 'satisfaction': 'очень_низкая'}, 'катастрофа'),
    ({'budget': 'очень_высокое', 'territory': 'очень_мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'territory': 'очень_мало', 'satisfaction': 'средняя'}, 'плохое'),
    ({'budget': 'очень_высокое', 'territory': 'очень_мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'очень_мало', 'satisfaction': 'очень_высокая'}, 'удовлетворительное'),

    # 5.2. Территории: МАЛО
    ({'budget': 'очень_высокое', 'territory': 'мало', 'satisfaction': 'очень_низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'territory': 'мало', 'satisfaction': 'низкая'}, 'плохое'),
    ({'budget': 'очень_высокое', 'territory': 'мало', 'satisfaction': 'средняя'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'мало', 'satisfaction': 'высокая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'мало', 'satisfaction': 'очень_высокая'}, 'хорошее'),

    # 5.3. Территории: СРЕДНЕ
    ({'budget': 'очень_высокое', 'territory': 'средне', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'средне', 'satisfaction': 'низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'средне', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'средне', 'satisfaction': 'высокая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'средне', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # 5.4. Территории: МНОГО
    ({'budget': 'очень_высокое', 'territory': 'много', 'satisfaction': 'очень_низкая'}, 'удовлетворительное'),
    ({'budget': 'очень_высокое', 'territory': 'много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'много', 'satisfaction': 'средняя'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'очень_высокое', 'territory': 'много', 'satisfaction': 'очень_высокая'}, 'отличное'),

    # 5.5. Территории: ОЧЕНЬ МНОГО
    ({'budget': 'очень_высокое', 'territory': 'очень_много', 'satisfaction': 'очень_низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'очень_много', 'satisfaction': 'низкая'}, 'хорошее'),
    ({'budget': 'очень_высокое', 'territory': 'очень_много', 'satisfaction': 'средняя'}, 'отличное'),
    ({'budget': 'очень_высокое', 'territory': 'очень_много', 'satisfaction': 'высокая'}, 'отличное'),
    ({'budget': 'очень_высокое', 'territory': 'очень_много', 'satisfaction': 'очень_высокая'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
public_spaces['Индекс качества общественного благоустройства'] = apply_fuzzy_convolution(public_spaces)
public_spaces

In [ ]:
public_spaces.describe()

## **Городской общественный транспорт**

In [ ]:
city_public_transport = df.iloc[:, 22:31]
city_public_transport

In [ ]:
city_public_transport.describe()

In [ ]:
city_public_transport_r1 = city_public_transport.drop(
    ['Дороги в нормативном состоянии. %.1',
     'Исполнение бюджета. %.3',
     'Регулируемые переходы. ед..1'], axis=1)
city_public_transport_r1.rename(columns={
    'Отремонтированные дороги. км.1': 'roads',
    'Пассажиропоток. тыс. поездок.1': 'passenger',
    'Рейсы по расписанию. %.1': 'schedule',
    'Средняя скорость на магистралях. км/ч.1': 'speed',
    'ДТП на 10 тыс. жителей.1': 'accident',
    'Срок устранения дефектов. суток.1': 'defect',
}, inplace=True)
city_public_transport_r1

In [ ]:
city_public_transport_r1.describe()

In [ ]:
# 1. Отремонтированные дороги (10.2-47.64 км) - ЧЕМ БОЛЬШЕ, ТЕМ ЛУЧШЕ
roads_var = LinguisticVariable('Отремонтированные дороги, км', 10, 48)
roads_var.add_term('мало', 'trimf', [10, 15, 25])           # 10-25 - мало
roads_var.add_term('средне', 'trimf', [20, 30, 40])         # 20-40 - средне
roads_var.add_term('много', 'trimf', [35, 42, 48])          # 35-48 - много

# 2. Пассажиропоток (33-113.8 тыс. поездок) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
passenger_var = LinguisticVariable('Пассажиропоток, тыс. поездок', 30, 115)
passenger_var.add_term('низкий', 'trimf', [30, 45, 65])     # 30-65 - низкий
passenger_var.add_term('средний', 'trimf', [55, 75, 95])    # 55-95 - средний
passenger_var.add_term('высокий', 'trimf', [85, 100, 115])  # 85-115 - высокий

# 3. Рейсы по расписанию (56.2-71.1%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
schedule_var = LinguisticVariable('Рейсы по расписанию, %', 55, 72)
schedule_var.add_term('низкое', 'trimf', [55, 57, 60])      # 55-60 - низкое
schedule_var.add_term('среднее', 'trimf', [58, 62, 66])     # 58-66 - среднее
schedule_var.add_term('высокое', 'trimf', [64, 68, 72])     # 64-72 - высокое

# 4. Средняя скорость на магистралях (22.4-31 км/ч) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
speed_var = LinguisticVariable('Средняя скорость, км/ч', 22, 31.5)
speed_var.add_term('низкая', 'trimf', [22, 23.5, 25.5])     # 22-25.5 - низкая
speed_var.add_term('средняя', 'trimf', [24.5, 26.5, 28.5])  # 24.5-28.5 - средняя
speed_var.add_term('высокая', 'trimf', [27.5, 29.5, 31.5])  # 27.5-31.5 - высокая

# 5. ДТП на 10 тыс. жителей (8.5-26.7) - ИНВЕРТИРОВАННЫЙ (чем меньше, тем лучше)
accident_var = LinguisticVariable('ДТП на 10 тыс. жителей', 8, 27)
accident_var.add_term('низкое', 'trimf', [8, 10, 13])       # 8-13 - мало ДТП (хорошо)
accident_var.add_term('среднее', 'trimf', [12, 15, 18])     # 12-18 - средне
accident_var.add_term('высокое', 'trimf', [16, 20, 27])     # 16-27 - много ДТП (плохо)

# 6. Срок устранения дефектов (3.2-6.6 сут) - ИНВЕРТИРОВАННЫЙ (чем меньше, тем лучше)
defect_var = LinguisticVariable('Срок устранения дефектов, сут', 3, 6.8)
defect_var.add_term('быстро', 'trimf', [3, 3.5, 4.2])       # 3-4.2 - быстро (хорошо)
defect_var.add_term('средне', 'trimf', [3.8, 4.5, 5.2])     # 3.8-5.2 - средне
defect_var.add_term('медленно', 'trimf', [4.8, 5.5, 6.8])   # 4.8-6.8 - медленно (плохо)

# 7. Выход: ИНДЕКС КАЧЕСТВА ДОРОГ
quality_var = LinguisticVariable('Индекс качества дорог', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('roads', roads_var)
fuzzy_system.add_input('passenger', passenger_var)
fuzzy_system.add_input('speed', speed_var)
fuzzy_system.add_input('schedule', schedule_var)
fuzzy_system.add_input('accident', accident_var)
fuzzy_system.add_input('defect', defect_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # roads: МАЛО, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
    # ==========================================
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
    ({'roads': 'мало', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
    # ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'катастрофа'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'удовлетворительное'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

# ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: СРЕДНЕ, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'средне', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'плохое'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'хорошее'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

# ==========================================
# roads: МНОГО, passenger: НИЗКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'низкий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: СРЕДНИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'средний', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: НИЗКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'удовлетворительное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'отличное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'низкое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: СРЕДНЕЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'среднее', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

# ==========================================
# roads: МНОГО, passenger: ВЫСОКИЙ, schedule: ВЫСОКОЕ
# ==========================================
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'медленно'}, 'хорошее'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'высокое', 'defect': 'быстро'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'низкая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'средне'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'средняя', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),

({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'медленно'}, 'отличное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'высокое', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'среднее', 'defect': 'быстро'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'медленно'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'средне'}, 'превосходное'),
({'roads': 'много', 'passenger': 'высокий', 'schedule': 'высокое', 'speed': 'высокая', 'accident': 'низкое', 'defect': 'быстро'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
city_public_transport_r1['Индекс качества дорог ГОТ'] = apply_fuzzy_convolution(city_public_transport_r1)
city_public_transport_r1

In [ ]:
city_public_transport_r1.describe()

In [ ]:
city_public_transport_r2 = road_transport_complex[['Дороги в нормативном состоянии. %',
                                                    'Исполнение бюджета. %',
                                                    'Регулируемые переходы. ед.']]
city_public_transport_r2.rename(columns={
    'Исполнение бюджета. %': 'budget',
    'Дороги в нормативном состоянии. %': 'norm',
    'Регулируемые переходы. ед.': 'crossing'}, inplace=True)
city_public_transport_r2

In [ ]:
city_public_transport_r2.describe()

In [ ]:
# 1. Нормативность дорог (40.9-52.3%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
norm_var = LinguisticVariable('Дороги в нормативном состоянии, %', 40, 53)
norm_var.add_term('очень_низкое', 'trimf', [40, 41.5, 43.5])      # 40-43.5 - очень низкое
norm_var.add_term('низкое', 'trimf', [42, 44, 46])                # 42-46 - низкое
norm_var.add_term('среднее', 'trimf', [44.5, 46.5, 48.5])         # 44.5-48.5 - среднее
norm_var.add_term('высокое', 'trimf', [47, 49, 51])               # 47-51 - высокое
norm_var.add_term('очень_высокое', 'trimf', [49.5, 51.5, 53])     # 49.5-53 - очень высокое

# 2. Исполнение бюджета (69.3-94.5%) - ЧЕМ ВЫШЕ, ТЕМ ЛУЧШЕ
budget_var = LinguisticVariable('Исполнение бюджета, %', 68, 96)
budget_var.add_term('очень_низкий', 'trimf', [68, 70.5, 75])       # 68-75 - очень низкий
budget_var.add_term('низкий', 'trimf', [73, 77, 81])               # 73-81 - низкий
budget_var.add_term('средний', 'trimf', [79, 83, 87])              # 79-87 - средний
budget_var.add_term('высокий', 'trimf', [85, 88.5, 92])            # 85-92 - высокий
budget_var.add_term('очень_высокий', 'trimf', [90, 93, 96])        # 90-96 - очень высокий

# 3. Регулируемые переходы (11-38 ед.) - ЧЕМ БОЛЬШЕ, ТЕМ ЛУЧШЕ
crossing_var = LinguisticVariable('Регулируемые переходы, ед.', 10, 40)
crossing_var.add_term('очень_мало', 'trimf', [10, 13, 18])         # 10-18 - очень мало
crossing_var.add_term('мало', 'trimf', [15, 20, 25])               # 15-25 - мало
crossing_var.add_term('средне', 'trimf', [22, 27, 32])             # 22-32 - средне
crossing_var.add_term('много', 'trimf', [28, 33, 36])              # 28-36 - много
crossing_var.add_term('очень_много', 'trimf', [34, 37, 40])        # 34-40 - очень много

# 4. Выход: ИНДЕКС БЛАГОПОЛУЧИЯ ДОРОГ
quality_var = LinguisticVariable('Индекс качества дорог', 1, 100)
quality_var.add_term('катастрофа', 'trimf', [1, 5, 20])
quality_var.add_term('плохое', 'trimf', [15, 28, 40])
quality_var.add_term('удовлетворительное', 'trimf', [35, 50, 65])
quality_var.add_term('хорошее', 'trimf', [60, 72, 84])
quality_var.add_term('отличное', 'trimf', [78, 88, 95])
quality_var.add_term('превосходное', 'trimf', [90, 97, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('norm', norm_var)
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('crossing', crossing_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # ==========================================
    # 1. НОРМАТИВНОСТЬ: ОЧЕНЬ НИЗКАЯ (40-43.5%)
    # ==========================================
    # 1.1. Бюджет: ОЧЕНЬ НИЗКИЙ (68-75%)
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 1.2. Бюджет: НИЗКИЙ (73-81%)
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 1.3. Бюджет: СРЕДНИЙ (79-87%)
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'средний', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 1.4. Бюджет: ВЫСОКИЙ (85-92%)
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 1.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ (90-96%)
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_низкое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 2. НОРМАТИВНОСТЬ: НИЗКАЯ (42-46%)
    # ==========================================
    # 2.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'плохое'),

    # 2.2. Бюджет: НИЗКИЙ
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 2.3. Бюджет: СРЕДНИЙ
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'средний', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 2.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # 2.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'низкое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # ==========================================
    # 3. НОРМАТИВНОСТЬ: СРЕДНЯЯ (44.5-48.5%)
    # ==========================================
    # 3.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'катастрофа'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'много'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 3.2. Бюджет: НИЗКИЙ
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 3.3. Бюджет: СРЕДНИЙ
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'средний', 'crossing': 'очень_много'}, 'хорошее'),

    # 3.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'высокий', 'crossing': 'очень_много'}, 'хорошее'),

    # 3.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'среднее', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'отличное'),

    # ==========================================
    # 4. НОРМАТИВНОСТЬ: ВЫСОКАЯ (47-51%)
    # ==========================================
    # 4.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'удовлетворительное'),

    # 4.2. Бюджет: НИЗКИЙ
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'хорошее'),

    # 4.3. Бюджет: СРЕДНИЙ
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'средний', 'crossing': 'очень_много'}, 'отличное'),

    # 4.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'отличное'),

    # 4.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'высокое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'превосходное'),

    # ==========================================
    # 5. НОРМАТИВНОСТЬ: ОЧЕНЬ ВЫСОКАЯ (49.5-53%)
    # ==========================================
    # 5.1. Бюджет: ОЧЕНЬ НИЗКИЙ
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'очень_мало'}, 'плохое'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'мало'}, 'плохое'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'средне'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'много'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_низкий', 'crossing': 'очень_много'}, 'хорошее'),

    # 5.2. Бюджет: НИЗКИЙ
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'много'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'низкий', 'crossing': 'очень_много'}, 'отличное'),

    # 5.3. Бюджет: СРЕДНИЙ
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'очень_мало'}, 'удовлетворительное'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'средне'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'средний', 'crossing': 'очень_много'}, 'отличное'),

    # 5.4. Бюджет: ВЫСОКИЙ
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'много'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'высокий', 'crossing': 'очень_много'}, 'превосходное'),

    # 5.5. Бюджет: ОЧЕНЬ ВЫСОКИЙ
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'очень_мало'}, 'хорошее'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'мало'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'средне'}, 'отличное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'много'}, 'превосходное'),
    ({'norm': 'очень_высокое', 'budget': 'очень_высокий', 'crossing': 'очень_много'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
city_public_transport_r2['Индекс благополучия дорог'] = apply_fuzzy_convolution(city_public_transport_r2)
city_public_transport_r2

In [ ]:
city_public_transport_r2.describe()

## **Парковки и безопасность движения**

In [ ]:
parking_and_traffic_security = df.iloc[:, 31:]
parking_and_traffic_security.rename(columns={
    'Исполнение бюджета. %.4': 'budget',
    'Отремонтированные дороги. км.2': 'roads',
    'Дороги в нормативном состоянии. %.2': 'norm',
    'Срок устранения дефектов. суток.2': 'defect',
}, inplace=True)
parking_and_traffic_security

In [ ]:
parking_and_traffic_security.describe()

In [ ]:
# 1.1. Исполнение бюджета (66.3-95.6%)
budget_var = LinguisticVariable('Бюджет, %', 60, 100)
budget_var.add_term('низкий', 'trimf', [60, 68, 76])
budget_var.add_term('средний', 'trimf', [74, 82, 90])
budget_var.add_term('высокий', 'trimf', [86, 94, 100])

# 1.2. Отремонтированные дороги (1.38-5.10 км)
roads_var = LinguisticVariable('Дороги, км', 1, 5.5)
roads_var.add_term('мало', 'trimf', [1, 1.8, 2.8])
roads_var.add_term('средне', 'trimf', [2.4, 3.2, 4.2])
roads_var.add_term('много', 'trimf', [3.8, 4.6, 5.5])

# 1.3. Дороги в нормативном состоянии (58.2-66.8%)
norm_var = LinguisticVariable('Нормативность, %', 57, 68)
norm_var.add_term('низкая', 'trimf', [57, 58.8, 61])
norm_var.add_term('средняя', 'trimf', [59.5, 62, 64.5])
norm_var.add_term('высокая', 'trimf', [63, 65.5, 68])

# 1.4. Срок устранения дефектов (17.3-39.5 сут) - ИНВЕРТИРОВАННЫЙ
defect_var = LinguisticVariable('Срок устранения, сут', 16, 42)
defect_var.add_term('быстро', 'trimf', [16, 19, 24])      # хорошо
defect_var.add_term('средне', 'trimf', [22, 28, 34])      # средне
defect_var.add_term('медленно', 'trimf', [32, 38, 42])    # плохо

# 1.5. Выход: ИНДЕКС ПАРКОВОК И БЕЗОПАСНОСТИ (МИНИМУМ = 10)
quality_var = LinguisticVariable('Индекс качества', 10, 100)
quality_var.add_term('катастрофа', 'trimf', [10, 15, 28])
quality_var.add_term('плохое', 'trimf', [23, 35, 48])
quality_var.add_term('удовлетворительное', 'trimf', [43, 55, 68])
quality_var.add_term('хорошее', 'trimf', [63, 73, 84])
quality_var.add_term('отличное', 'trimf', [78, 87, 95])
quality_var.add_term('превосходное', 'trimf', [90, 96, 100])

In [ ]:
# Указание правил
fuzzy_system = FuzzySystem()
fuzzy_system.add_input('budget', budget_var)
fuzzy_system.add_input('roads', roads_var)
fuzzy_system.add_input('norm', norm_var)
fuzzy_system.add_input('defect', defect_var)
fuzzy_system.set_output('quality_index', quality_var)

rules = [
    # ==========================================
    # 1. БЮДЖЕТ: НИЗКИЙ
    # ==========================================
    # 1.1. Дороги: МАЛО
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'медленно'}, 'катастрофа'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'средне'}, 'катастрофа'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'быстро'}, 'плохое'),

    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'средне'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'быстро'}, 'удовлетворительное'),

    # 1.2. Дороги: СРЕДНЕ
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'медленно'}, 'катастрофа'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'средне'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'быстро'}, 'плохое'),

    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'быстро'}, 'хорошее'),

    # 1.3. Дороги: МНОГО
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'низкая', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'низкая', 'defect': 'средне'}, 'плохое'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'низкая', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'низкий', 'roads': 'много', 'norm': 'средняя', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'средняя', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'средняя', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'низкий', 'roads': 'много', 'norm': 'высокая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'высокая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'низкий', 'roads': 'много', 'norm': 'высокая', 'defect': 'быстро'}, 'хорошее'),

    # ==========================================
    # 2. БЮДЖЕТ: СРЕДНИЙ
    # ==========================================
    # 2.1. Дороги: МАЛО
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'низкая', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'низкая', 'defect': 'средне'}, 'плохое'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'низкая', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'средний', 'roads': 'мало', 'norm': 'средняя', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'средняя', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'средняя', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'средний', 'roads': 'мало', 'norm': 'высокая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'высокая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'мало', 'norm': 'высокая', 'defect': 'быстро'}, 'хорошее'),

    # 2.2. Дороги: СРЕДНЕ
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'низкая', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'низкая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'низкая', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'средний', 'roads': 'средне', 'norm': 'средняя', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'средняя', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'средняя', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'средний', 'roads': 'средне', 'norm': 'высокая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'высокая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'средний', 'roads': 'средне', 'norm': 'высокая', 'defect': 'быстро'}, 'хорошее'),

    # 2.3. Дороги: МНОГО
    ({'budget': 'средний', 'roads': 'много', 'norm': 'низкая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'низкая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'низкая', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'средний', 'roads': 'много', 'norm': 'средняя', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'средняя', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'средняя', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'средний', 'roads': 'много', 'norm': 'высокая', 'defect': 'медленно'}, 'хорошее'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'высокая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'средний', 'roads': 'много', 'norm': 'высокая', 'defect': 'быстро'}, 'отличное'),

    # ==========================================
    # 3. БЮДЖЕТ: ВЫСОКИЙ
    # ==========================================
    # 3.1. Дороги: МАЛО
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'медленно'}, 'плохое'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'низкая', 'defect': 'быстро'}, 'удовлетворительное'),

    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'средняя', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'мало', 'norm': 'высокая', 'defect': 'быстро'}, 'хорошее'),

    # 3.2. Дороги: СРЕДНЕ
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'средне'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'низкая', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'средняя', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'медленно'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'средне', 'norm': 'высокая', 'defect': 'быстро'}, 'отличное'),

    # 3.3. Дороги: МНОГО
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'низкая', 'defect': 'медленно'}, 'удовлетворительное'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'низкая', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'низкая', 'defect': 'быстро'}, 'хорошее'),

    ({'budget': 'высокий', 'roads': 'много', 'norm': 'средняя', 'defect': 'медленно'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'средняя', 'defect': 'средне'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'средняя', 'defect': 'быстро'}, 'отличное'),

    ({'budget': 'высокий', 'roads': 'много', 'norm': 'высокая', 'defect': 'медленно'}, 'хорошее'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'высокая', 'defect': 'средне'}, 'отличное'),
    ({'budget': 'высокий', 'roads': 'много', 'norm': 'высокая', 'defect': 'быстро'}, 'превосходное'),
]

for premise, consequent in rules:
    fuzzy_system.add_rule(premise, consequent)

In [ ]:
parking_and_traffic_security['Индекс качества парковок и безопасности движения'] = apply_fuzzy_convolution(parking_and_traffic_security)
parking_and_traffic_security

In [ ]:
parking_and_traffic_security.describe()